# Week 12-2 · TBP-02 — REST API for Algorithmic Trading

**Part 2 of the automation module.** Last time (TBP-01) we automated trading with the
**IB TWS API** — a *custom* Python-library API (`EClient` / `EWrapper`). This time we learn a
**second, broker-agnostic** way: the **REST API**.

> **Why a second approach?** (1) Knowing several ways lets you pick the *best* one for a job.
> (2) REST has become the **de-facto standard** — almost every broker that supports algo
> trading (especially post-COVID) exposes a REST API: Indian, international, crypto, and FX
> brokers alike. Learn it once and you can talk to *virtually any* broker.

**What REST is NOT:**
- **Not for HFT** — it runs over the internet (seconds of latency); good for **LFT / MFT**
  (down to ~1 second). HFT firms use **FIX** and **FPGA** hardware instead.
- **Not uniform** — each broker exposes a *different* set of operations; always read the docs.
- **Not a replacement** — it *complements* the web/desktop terminal (which you keep for monitoring).

Since we have no live broker account here, we build a **fully offline MOCK broker** that mimics
`requests.get` / `requests.post`, real **status codes**, and **JSON** — so the exact code
pattern from the lecture *runs*. We feed it real NIFTY bars from `historical_sample.csv`.

## 1. API vs REST — the definitions

- **API** = Application Programming Interface: a *programmable* bridge that lets a **client**
  talk to a **server** in a structured way.
- **REST** = **RE**presentational **S**tate **T**ransfer. It is not a library — it is a set of
  **design principles** (uniform interface, **stateless**, cacheable, client–server). Any API
  built to follow them is "a REST API." As the *end user* we don't care about the principles;
  we care that REST APIs come as **HTTP URLs** (unlike the IB library, which was a Python package).

Intuition: the client **requests** the state of some variable (PnL, positions, price); the
server **reads** that state and **transfers** a **representation** of it back. Hence
*Representational State Transfer*. REST is **synchronous**: request → wait → response → act.

## 2. The anatomy of a request — five pieces

Every REST request to a broker needs **five** ingredients. Miss one and it fails:

| # | Piece | What it is |
|---|-------|-----------|
| 1 | **Base URL** | the broker's web-server address, e.g. `https://www.brokerapi.com` (docs, page 1) |
| 2 | **Endpoint** | the "department" for one operation, e.g. `/positions`, `/place_order`, `/pnl` |
| 3 | **HTTP method** | the *intention*: **GET** (fetch) or **POST** (send/create) — docs tell you which |
| 4 | **Headers** | info *about the client* (auth token, content type) — a Python **dict** |
| 5 | **Payload / body** | info about the *request itself* (symbol, qty, price) — a dict, sent as **JSON** |

The full URL = **base URL + endpoint**, e.g. `https://www.brokerapi.com/place_order`.

In [1]:
import json, time, warnings
warnings.filterwarnings("ignore")
import pandas as pd

# ---- A MOCK broker REST server (mimics IB Client-Portal Web API), fully offline ----
bars = pd.read_csv("historical_sample.csv", parse_dates=["Date"]).tail(10).reset_index(drop=True)

class MockResponse:
    """Looks like a requests.Response: .status_code, .content, .json()."""
    def __init__(self, status_code, payload):
        self.status_code = status_code
        self._payload = payload
        self.content = json.dumps(payload).encode()   # server returns JSON bytes
    def json(self):
        return json.loads(self.content)               # client parses JSON -> dict

class MockBroker:
    """A tiny broker web-server with endpoints, auth state, and JSON I/O."""
    def __init__(self):
        self.authenticated = False
        self.last_seen = 0
        self._order_id = 1000
        self.orders = {}
        self.positions = {"NIFTY": 50}
    def _touch(self):                                  # IB disconnects if idle too long
        self.last_seen = time.time()
    def handle(self, method, endpoint, headers, payload):
        self._touch()
        # ---- routing table: (METHOD, endpoint) -> behaviour ----
        if endpoint == "/iserver/auth/status" and method == "POST":
            return MockResponse(200, {"authenticated": self.authenticated,
                                       "connected": self.authenticated})
        if endpoint == "/login" and method == "POST":
            self.authenticated = True
            return MockResponse(200, {"message": "client login succeeds"})
        if endpoint == "/iserver/reauthenticate" and method == "POST":
            self.authenticated = True
            return MockResponse(200, {"message": "re-authentication triggered"})
        if not self.authenticated:
            return MockResponse(401, {"error": "not authenticated"})
        if endpoint == "/pnl" and method == "GET":
            return MockResponse(200, {"realized": 12450.0, "unrealized": -3200.0})
        if endpoint == "/positions" and method == "GET":
            return MockResponse(200, {"positions": self.positions})
        if endpoint == "/historical" and method == "GET":
            data = [{"date": str(d.date()), "close": float(c)}
                    for d, c in zip(bars["Date"], bars["Close"])]
            return MockResponse(200, {"symbol": payload.get("symbol"), "bars": data})
        if endpoint == "/place_order" and method == "POST":
            self._order_id += 1
            self.orders[self._order_id] = {**payload, "status": "Submitted"}
            return MockResponse(200, {"order_id": self._order_id, "status": "Submitted"})
        # unknown endpoint -> client-side error
        return MockResponse(404, {"error": "page not found", "endpoint": endpoint})

broker = MockBroker()

# ---- a drop-in 'requests'-like shim so the lecture code reads identically ----
class requests:
    @staticmethod
    def get(url, headers=None, json=None, verify=False):
        ep = "/" + url.split("/v1/api/")[1]
        return broker.handle("GET", ep, headers or {}, json or {})
    @staticmethod
    def post(url, headers=None, json=None, verify=False):
        ep = "/" + url.split("/v1/api/")[1]
        return broker.handle("POST", ep, headers or {}, json or {})

print("Mock broker ready. 10 NIFTY bars loaded:", bars["Date"].min().date(), "->", bars["Date"].max().date())

Mock broker ready. 10 NIFTY bars loaded: 2020-11-13 -> 2020-11-27


## 3. First request — check authentication status

The IB REST API (called the **Web API** / Client-Portal Web API) runs through a local
**gateway** at `https://localhost:5000/v1/api`. The docs say the auth-status endpoint uses
**POST**, needs **no headers or payload**. We also use `verify=False` — IB's local gateway has
**no SSL certificate** (specific to IB).

In [2]:
BASE_URL = "https://localhost:5000/v1/api"

def auth_status():
    endpoint = "/iserver/auth/status"
    url = BASE_URL + endpoint                 # piece 1 + piece 2
    print("URL:", url)
    headers, payload = {}, {}                 # pieces 4 & 5: none needed here
    r = requests.post(url, headers=headers, json=payload, verify=False)  # piece 3: POST
    status = r.status_code                     # response part 1: status code
    content = json.loads(r.content)            # response part 2: content (JSON -> dict)
    print("status code:", status)
    print("content:", content)
    return content

auth_status()   # before logging in -> authenticated False (exactly the lecture's gotcha)

URL: https://localhost:5000/v1/api/iserver/auth/status
status code: 200
content: {'authenticated': False, 'connected': False}


**The gotcha (a feature, not a bug):** before login we're *not* authenticated. IB also
**disconnects you if you stay idle** — in the lecture, the instructor logged in, spent minutes
explaining the code, and got auto-disconnected by the time he re-ran it.

In [3]:
# Log in (the gateway browser step), then re-check
def login():
    r = requests.post(BASE_URL + "/login", json={}, verify=False)
    print(r.status_code, json.loads(r.content))

login()
auth_status()   # now authenticated True / connected True

200 {'message': 'client login succeeds'}
URL: https://localhost:5000/v1/api/iserver/auth/status
status code: 200
content: {'authenticated': True, 'connected': True}


## 4. Reading the status code — the three families

Always check the **status code first**, then the content. You don't memorize them — you read
the **first digit**:

| Starts with | Meaning | Whose fault |
|-------------|---------|-------------|
| **2xx** | success — server understood and processed | — |
| **4xx** | **client** error — bad URL/endpoint/method/headers/payload (e.g. **404** page not found) | **you** |
| **5xx** | **server** error — your request was fine but the server can't cope right now | the broker |

These codes are an **HTTP-consortium** standard — universal across the entire web, not
broker-specific.

In [4]:
# A deliberately wrong endpoint -> 4xx client error
def bad_request():
    url = BASE_URL + "/iserver/auth/statuss"   # typo!
    r = requests.post(url, json={}, verify=False)
    print("URL:", url)
    print("status code:", r.status_code, "->", "CLIENT error (4xx)" if 400 <= r.status_code < 500 else "")
    print("content:", json.loads(r.content))

bad_request()

URL: https://localhost:5000/v1/api/iserver/auth/statuss
status code: 404 -> CLIENT error (4xx)
content: {'error': 'page not found', 'endpoint': '/iserver/auth/statuss'}


## 5. GET vs POST — fetch PnL, fetch positions

**GET** = *get me information*. The IB REST API exposes one endpoint per operation; you discover
them by scrolling the docs. Here we fetch **PnL** and **positions** (both GET, after auth).

In [5]:
def fetch_pnl():
    url = BASE_URL + "/pnl"
    r = requests.get(url, verify=False)        # GET: we want information
    print("PnL status", r.status_code, "->", r.json())
    return r.json()

def fetch_positions():
    r = requests.get(BASE_URL + "/positions", verify=False)
    print("Positions status", r.status_code, "->", r.json())
    return r.json()

pnl = fetch_pnl()
pos = fetch_positions()
print("\nNet PnL:", pnl["realized"] + pnl["unrealized"])

PnL status 200 -> {'realized': 12450.0, 'unrealized': -3200.0}
Positions status 200 -> {'positions': {'NIFTY': 50}}

Net PnL: 9250.0


## 6. POST with headers + payload — place an order

**POST** = *here is new information to act on*. Now all five pieces matter. The **payload**
(a.k.a. body) carries the order details — symbol, side, quantity, type — and is sent as **JSON**
via `json.dumps` under the hood. **Headers** carry client info (e.g. an auth token / content type).

In [6]:
def place_order(symbol, side, qty, order_type="MKT"):
    endpoint = "/place_order"
    url = BASE_URL + endpoint
    headers = {"Content-Type": "application/json", "Connection": "keep-alive"}   # about the client
    payload = {"symbol": symbol, "side": side, "quantity": qty, "orderType": order_type}  # about the request
    print("URL:", url)
    print("payload (dict):", payload)
    print("payload (JSON):", json.dumps(payload))     # what actually crosses the wire
    r = requests.post(url, headers=headers, json=payload, verify=False)
    out = r.json()
    print("status code:", r.status_code, "->", out)
    return out

order = place_order("NIFTY", "BUY", 75)
print("\nBroker assigned order_id:", order["order_id"])

URL: https://localhost:5000/v1/api/place_order
payload (dict): {'symbol': 'NIFTY', 'side': 'BUY', 'quantity': 75, 'orderType': 'MKT'}
payload (JSON): {"symbol": "NIFTY", "side": "BUY", "quantity": 75, "orderType": "MKT"}
status code: 200 -> {'order_id': 1001, 'status': 'Submitted'}

Broker assigned order_id: 1001


## 7. Why JSON? — the common language between client and server

Our headers/payloads are **Python dicts**; the server might be written in C++ (which has no
"dict"). JSON (**JavaScript Object Notation** — nothing to do with JavaScript) is a **neutral
text format every language understands**. The `json` library bridges it:

- `json.dumps(dict)` → JSON string (client → server)
- `json.loads(json)` → dict (server → client)

In [7]:
payload = {"symbol": "NIFTY", "side": "BUY", "quantity": 75}
as_json = json.dumps(payload)          # dict -> JSON text (to send)
back = json.loads(as_json)             # JSON text -> dict (when received)
print("python dict :", payload, type(payload).__name__)
print("JSON string :", as_json, type(as_json).__name__)
print("parsed back :", back, type(back).__name__)
print("round-trips equal:", payload == back)

python dict : {'symbol': 'NIFTY', 'side': 'BUY', 'quantity': 75} dict
JSON string : {"symbol": "NIFTY", "side": "BUY", "quantity": 75} str
parsed back : {'symbol': 'NIFTY', 'side': 'BUY', 'quantity': 75} dict
round-trips equal: True


## 8. Historical data + a tiny modular "library" pattern

The instructor's design: put **one function per endpoint** in a `cp_web_api_functions.py`
file (your own mini-library), then `import` them into an `entry_point.py` where your **strategy**
lives — so strategy code never touches broker-specific plumbing. We mirror that: `fetch_historical`
is a reusable function, and the cell after it is the strategy entry point.

In [8]:
def fetch_historical(symbol):
    r = requests.get(BASE_URL + "/historical", json={"symbol": symbol}, verify=False)
    if r.status_code != 200:                  # ALWAYS check status before content
        print("request failed:", r.status_code); return None
    return pd.DataFrame(r.json()["bars"])

hist = fetch_historical("NIFTY")
print(hist.to_string(index=False))

      date        close
2020-11-13 12719.950195
2020-11-17 12874.200195
2020-11-18 12938.250000
2020-11-19 12771.700195
2020-11-20 12859.049805
2020-11-23 12926.450195
2020-11-24 13055.150391
2020-11-25 12858.400391
2020-11-26 12987.000000
2020-11-27 12968.950195


In [9]:
# entry_point.py style: strategy logic only, no broker plumbing
def simple_strategy(symbol):
    df = fetch_historical(symbol)             # reuse the library function
    df["ret"] = df["close"].pct_change()
    signal = "BUY" if df["ret"].iloc[-1] > 0 else "SELL"
    print(f"last close {df['close'].iloc[-1]:.1f}, last move {df['ret'].iloc[-1]*100:+.2f}% -> signal {signal}")
    return place_order(symbol, signal, 75)

result = simple_strategy("NIFTY")

last close 12969.0, last move -0.14% -> signal SELL
URL: https://localhost:5000/v1/api/place_order
payload (dict): {'symbol': 'NIFTY', 'side': 'SELL', 'quantity': 75, 'orderType': 'MKT'}
payload (JSON): {"symbol": "NIFTY", "side": "SELL", "quantity": 75, "orderType": "MKT"}
status code: 200 -> {'order_id': 1002, 'status': 'Submitted'}


## 9. Staying connected — the heartbeat ("continuous tickle")

IB **disconnects an idle session**. Two fixes: call the **`/reauthenticate`** endpoint, or run a
**continuous tickle** — a background loop hitting a `validate`/`tickle` endpoint every ~55s on a
**separate thread** (just like the listener thread in TBP-01). You may **not** automate the
*initial login* — that's a regulatory restriction, so brokers require a manual log-in.

In [10]:
def reauthenticate():
    r = requests.post(BASE_URL + "/iserver/reauthenticate", json={}, verify=False)
    print(r.status_code, r.json())

# simulate going idle, getting disconnected, then re-authenticating
broker.authenticated = False
print("after idle:", auth_status()["authenticated"])
reauthenticate()
print("after tickle:", auth_status()["authenticated"])

URL: https://localhost:5000/v1/api/iserver/auth/status
status code: 200
content: {'authenticated': False, 'connected': False}
after idle: False
200 {'message': 're-authentication triggered'}
URL: https://localhost:5000/v1/api/iserver/auth/status
status code: 200
content: {'authenticated': True, 'connected': True}
after tickle: True


## 10. REST vs the IB TWS API — and the one-paragraph version

| | **IB TWS API** (TBP-01) | **REST / Web API** (TBP-02) |
|---|---|---|
| Form | custom Python **library** (`EClient`/`EWrapper`) | **HTTP URLs** |
| Communication | **asynchronous** (listener loop) | **synchronous** (request → wait → response) |
| Reach | IB only | **any broker** that supports REST |
| You build | use IB's classes | **your own** request/response functions |

**One-paragraph version.** A **REST API** is the broker-agnostic, de-facto-standard way to
automate trading over HTTP — great for LFT/MFT, never for HFT (use FIX/FPGA). REST =
*Representational State Transfer*: a **synchronous** client request for the **state** of some
variable, returned as a representation. Every request needs **five pieces** — base URL, endpoint,
**HTTP method** (GET=fetch, POST=send), **headers** (about the client), **payload/body** (about
the request, sent as **JSON**). The response has two parts: a **status code** (read the first
digit — **2xx** OK, **4xx** your error, **5xx** server error) and the **content**. Use **JSON**
(`dumps`/`loads`) as the common language between Python and a server written in any tongue. Keep
the session alive with a **heartbeat/tickle** thread (you can't auto-login). Wrap one function per
endpoint into your own little library so your **strategy** code stays clean — exactly what we did
on a mock broker fed with real NIFTY bars.

# Additive Source Validation Section

The following cells are appended from `Week 12-2 TBP-02 Rest API_resource_addendum_practice.ipynb`. They do not modify the original mock broker lesson; they add coverage for the official source scripts, WebSocket examples, heartbeat handling, and corrected RSI strategy logic.

## 1. Load the additive reference files

The source scripts were copied under new `*_source` filenames. The tables below map those live examples into offline-safe practice objects.

In [11]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

code_dir = Path.cwd()
inventory = pd.read_csv(code_dir / "tbp02_source_file_inventory.csv")
endpoints = pd.read_csv(code_dir / "tbp02_endpoint_reference.csv")
status_ref = pd.read_csv(code_dir / "tbp02_status_code_reference.csv")
channels = pd.read_csv(code_dir / "tbp02_rest_websocket_reference.csv")

print("Source files copied:", len(inventory[inventory["added_copy"] != "not copied"]))
print("Endpoint/channel rows:", len(endpoints))
print(endpoints[["function", "method", "endpoint"]].head(10).to_string(index=False))

Source files copied: 8
Endpoint/channel rows: 19
              function method                        endpoint
          validate_sso    GET                    sso/validate
                logout   POST                          logout
        reauthenticate   POST          iserver/reauthenticate
           auth_status   POST             iserver/auth/status
                tickle   POST                          tickle
fetch_contract_details    GET   iserver/contract/{conid}/info
 fetch_historical_data    GET      iserver/marketdata/history
             fetch_ltp    GET     iserver/marketdata/snapshot
        fetch_accounts    GET                iserver/accounts
       get_account_pnl    GET iserver/account/pnl/partitioned


## 2. Status-first REST request handling

The lecture rule is to read the HTTP status family first, then parse content only when the response is usable. The mock below includes success, client error, and server error cases without making any network request.

In [12]:
class OfflineResponse:
    def __init__(self, status_code, payload):
        self.status_code = status_code
        self.content = json.dumps(payload).encode("utf-8")
    def json(self):
        return json.loads(self.content)

def interpret_response(operation, response):
    family = f"{str(response.status_code)[0]}xx"
    payload = response.json()
    if 200 <= response.status_code < 300:
        action = "parse content"
    elif 400 <= response.status_code < 500:
        action = "fix client request or authentication"
    elif response.status_code >= 500:
        action = "log, pause, and retry only under policy"
    else:
        action = "check broker documentation"
    return {"operation": operation, "status_code": response.status_code, "family": family, "client_action": action, "content": payload}

responses = [
    ("auth_status", OfflineResponse(200, {"authenticated": True, "connected": True})),
    ("bad_endpoint", OfflineResponse(404, {"error": "page not found"})),
    ("gateway_down", OfflineResponse(503, {"error": "gateway unavailable"})),
]
status_demo = pd.DataFrame([interpret_response(op, res) for op, res in responses])
print(status_demo[["operation", "status_code", "family", "client_action"]].to_string(index=False))

   operation  status_code family                           client_action
 auth_status          200    2xx                           parse content
bad_endpoint          404    4xx    fix client request or authentication
gateway_down          503    5xx log, pause, and retry only under policy


## 3. One source-style request builder

The source strategy adds `make_request(method, url, headers, payload, qs_params)`. Offline, the same idea is a request record: base URL, endpoint, method, headers, and payload/params are separated before the mock response is processed.

In [13]:
BASE_URL = "https://localhost:5000/v1/api"

def build_request(method, endpoint, headers=None, payload=None, params=None):
    return {
        "base_url": BASE_URL,
        "endpoint": endpoint,
        "url": BASE_URL + "/" + endpoint.lstrip("/"),
        "method": method,
        "headers": headers or {},
        "payload": payload or {},
        "params": params or {},
    }

order_request = build_request(
    "POST",
    "iserver/account/DUXXXX/orders",
    headers={"Content-Type": "application/json"},
    payload={"orders": [{"conid": 56987333, "orderType": "MKT", "side": "BUY", "quantity": 1, "price": 0, "tif": "DAY"}]},
)
print(json.dumps(order_request, indent=2))

{
  "base_url": "https://localhost:5000/v1/api",
  "endpoint": "iserver/account/DUXXXX/orders",
  "url": "https://localhost:5000/v1/api/iserver/account/DUXXXX/orders",
  "method": "POST",
  "headers": {
    "Content-Type": "application/json"
  },
  "payload": {
    "orders": [
      {
        "conid": 56987333,
        "orderType": "MKT",
        "side": "BUY",
        "quantity": 1,
        "price": 0,
        "tif": "DAY"
      }
    ]
  },
  "params": {}
}


## 4. WebSocket message parsing without a socket

REST calls return one response. WebSockets push repeated messages after a subscription such as `smd+contract+fields`. These samples mirror the IB field codes and Alpaca quote shape from the source files.

In [14]:
messages = pd.read_csv(code_dir / "tbp02_mock_websocket_messages.csv")

def parse_ib_market_data(row):
    return {
        "source": row["source"],
        "topic": row["raw_topic"],
        "last_price": float(row["31"]) if pd.notna(row.get("31")) else np.nan,
        "bid": float(row["84"]) if pd.notna(row.get("84")) else np.nan,
        "ask": float(row["86"]) if pd.notna(row.get("86")) else np.nan,
    }

ib_quote = parse_ib_market_data(messages.iloc[0])
alpaca_quote = messages[messages["source"] == "Alpaca"].iloc[0][["symbol", "bid", "ask"]].to_dict()
print("IB parsed quote:", ib_quote)
print("Alpaca parsed quote:", alpaca_quote)

IB parsed quote: {'source': 'IB', 'topic': 'smd+56987333', 'last_price': 18123.5, 'bid': 18122.0, 'ask': 18124.0}
Alpaca parsed quote: {'symbol': 'BTCUSD', 'bid': 63250.1, 'ask': 63252.4}


## 5. Corrected RSI decision table and offline order audit

The source RSI script is preserved as copied source. The addendum corrects the close-position branch by assigning `order_action = 'SELL'` or `order_action = 'BUY'` when exiting existing positions.

In [15]:
decisions = pd.read_csv(code_dir / "tbp02_rsi_strategy_decision_table.csv")
orders = pd.read_csv(code_dir / "tbp02_mock_order_audit.csv")
metrics = pd.read_csv(code_dir / "tbp02_offline_rsi_strategy_metrics.csv")
print(decisions.to_string(index=False))
print("\nMock orders generated:", len(orders))
print(orders[["order_id", "date", "side", "prior_position", "target_position", "reason"]].tail(8).to_string(index=False))
print("\nKey metrics:")
print(metrics[metrics["metric"].isin(["mock_order_count", "offline_rsi_total_return_pct", "buy_hold_total_return_pct", "offline_rsi_max_drawdown_pct"])].to_string(index=False))

current_position         rsi_condition target_position correct_order_action                                                             source_status
               0              RSI < 30               1                  BUY                       source logic already assigns BUY for new long entry
               0              RSI > 70              -1                 SELL                     source logic already assigns SELL for new short entry
               1              RSI > 70               0                 SELL source line uses order_action == SELL; corrected addendum uses assignment
              -1              RSI < 30               0                  BUY  source line uses order_action == BUY; corrected addendum uses assignment
             any threshold not crossed       unchanged                 none                                                     hold current position

Mock orders generated: 13
    order_id       date side  prior_position  target_position            

## 6. Heartbeat schedule

The initial login remains manual. After that, the strategy should keep the gateway alive with periodic `tickle` or reauthentication calls between data and order events.

In [16]:
heartbeat = pd.read_csv(code_dir / "tbp02_heartbeat_schedule.csv")
request_log = pd.read_csv(code_dir / "tbp02_mock_request_log.csv")
print(heartbeat.to_string(index=False))
print("\nStatus families in the mock request log:")
print(request_log.groupby("status_family").size().to_string())

 seconds_from_start                     operation                                                             reason
                  0 manual login already complete keep local gateway session from going idle between strategy events
                 55        POST /tickle heartbeat keep local gateway session from going idle between strategy events
                110        POST /tickle heartbeat keep local gateway session from going idle between strategy events
                165        POST /tickle heartbeat keep local gateway session from going idle between strategy events
                220        POST /tickle heartbeat keep local gateway session from going idle between strategy events
                275        POST /tickle heartbeat keep local gateway session from going idle between strategy events
                330        POST /tickle heartbeat keep local gateway session from going idle between strategy events

Status families in the mock request log:
status_family
2xx    6